# Distance Metrics Analysis (ETL Pipeline)

Extracts, transforms, and analyzes distance metric tables (`*distance_metrics_*.csv`) produced by `notebooks/single_region_full_validation.ipynb` across multiple single-region validation runs.

### Pipeline Highlights:
- Configurable input paths to target validation execution folders.
- Dynamic extraction of kinematic region `(ET, ETA)` from filenames.
- Lineage tracking of origin `ExecutionFolder`.
- Consolidated multi-region analysis and export.

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as mh

mh.style.use("ATLAS")

from src.core.Validation.DistanceMetricsETL import DistanceMetricsETL

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/pmourafr/egamma_master


## 1. ETL Configuration


In [26]:
VALIDATION_FOLDERS: list[Path] = [
    PROJECT_ROOT / "notebooks/Tables",
]

# Analysis of distance metrics

## 1. ETL Configuration

Specify the validation folders containing the `Tables/` directory, or provide folder paths directly.

In [29]:
FILE_PATTERN: str = "*distance_metrics_*.csv"

print(f"Configured {len(VALIDATION_FOLDERS)} folders to scan:")
for folder in VALIDATION_FOLDERS:
    print(f" - {folder}")

Configured 1 folders to scan:
 - /home/pmourafr/egamma_master/notebooks/Tables


## 2. Execute Extraction & Consolidation

In [34]:
etl = DistanceMetricsETL(folder_paths=VALIDATION_FOLDERS, file_pattern=FILE_PATTERN)
print(f"Discovered {len(etl.discovered_files)} distance metric CSV files.")

df = etl.extract_and_consolidate()
print(f"Consolidated DataFrame shape: {df.shape}")
display(df[
    (df["Feature"]=="Eratio") &
    (~df["Reference"].str.contains('Athena')) 
    ])

Discovered 4 distance metric CSV files.
Consolidated DataFrame shape: (144, 12)


,Feature,Reference,Strategy,Comparison,Ref Count,Selected Count,JS Divergence,JS Uncertainty,ET,ETA,ExecutionFolder,MetricType
33,Eratio,True Background,Model V9 (100 Rings),Model V9 (100 Rings) vs True Background,10384,1446,0.223816,0.010282,1,1,notebooks,vs_true_background
34,Eratio,True Background,Athena Loose Cut,Athena Loose Cut vs True Background,10384,4309,0.007109,0.001315,1,1,notebooks,vs_true_background
35,Eratio,True Background,Model V9 (50 Rings),Model V9 (50 Rings) vs True Background,10384,1514,0.207126,0.009078,1,1,notebooks,vs_true_background
69,Eratio,True Background,Model V9 (100 Rings) [T(mu)],Model V9 (100 Rings) [T(mu)] vs True Background,10384,1486,0.217338,0.009435,1,1,notebooks,vs_true_background_pileup
70,Eratio,True Background,Athena Loose Cut,Athena Loose Cut vs True Background,10384,4309,0.007109,0.001315,1,1,notebooks,vs_true_background_pileup
71,Eratio,True Background,Model V9 (50 Rings) [T(mu)],Model V9 (50 Rings) [T(mu)] vs True Background,10384,1585,0.196813,0.008957,1,1,notebooks,vs_true_background_pileup
105,Eratio,True Signal,Model V9 (100 Rings),Model V9 (100 Rings) vs True Signal,3994,3247,0.019469,0.002953,1,1,notebooks,vs_true_signal
106,Eratio,True Signal,Athena Loose Cut,Athena Loose Cut vs True Signal,3994,3106,0.021001,0.002826,1,1,notebooks,vs_true_signal
107,Eratio,True Signal,Model V9 (50 Rings),Model V9 (50 Rings) vs True Signal,3994,3247,0.020365,0.002672,1,1,notebooks,vs_true_signal
141,Eratio,True Signal,Model V9 (100 Rings) [T(mu)],Model V9 (100 Rings) [T(mu)] vs True Signal,3994,3255,0.019508,0.002407,1,1,notebooks,vs_true_signal_pileup


## 3. Executive Summary by Region & Run

In [35]:
summary_df = etl.get_summary(df)
if not summary_df.empty:
    display(summary_df)
else:
    print("No records found to summarize.")

,ExecutionFolder,ET,ETA,MetricType,RecordCount
0,notebooks,1,1,vs_true_background,36
1,notebooks,1,1,vs_true_background_pileup,36
2,notebooks,1,1,vs_true_signal,36
3,notebooks,1,1,vs_true_signal_pileup,36


## 4. Query & Inspect Specific Metrics & Features

In [36]:
if not df.empty:
    print("Available Metric Types:", df["MetricType"].unique())
    print("Available Features:", df["Feature"].unique() if "Feature" in df.columns else "N/A")

    # Example pivot / inspection
    metric_type_target = df["MetricType"].unique()[0]
    subset = df[df["MetricType"] == metric_type_target]
    print(f"\nSample data for {metric_type_target}:")
    display(subset.head())
else:
    print("Consolidated DataFrame is empty.")

Available Metric Types: ['vs_true_background' 'vs_true_background_pileup' 'vs_true_signal'
 'vs_true_signal_pileup']
Available Features: ['trig_L2_calo_et' 'trig_L2_calo_eta' 'trig_L2_calo_phi'
 'trig_L2_calo_e237' 'trig_L2_calo_e277' 'trig_L2_calo_fracs1'
 'trig_L2_calo_weta2' 'trig_L2_calo_ehad1' 'trig_L2_calo_emaxs1'
 'trig_L2_calo_e2tsts1' 'trig_L2_calo_wstot' 'Eratio']

Sample data for vs_true_background:


,Feature,Reference,Strategy,Comparison,Ref Count,Selected Count,JS Divergence,JS Uncertainty,ET,ETA,ExecutionFolder,MetricType
0,trig_L2_calo_et,True Background,Model V9 (100 Rings),Model V9 (100 Rings) vs True Background,10384,1446,0.005162,0.002339,1,1,notebooks,vs_true_background
1,trig_L2_calo_et,True Background,Athena Loose Cut,Athena Loose Cut vs True Background,10384,4309,0.002486,0.000915,1,1,notebooks,vs_true_background
2,trig_L2_calo_et,True Background,Model V9 (50 Rings),Model V9 (50 Rings) vs True Background,10384,1514,0.006024,0.002073,1,1,notebooks,vs_true_background
3,trig_L2_calo_eta,True Background,Model V9 (100 Rings),Model V9 (100 Rings) vs True Background,10384,1446,0.004409,0.001769,1,1,notebooks,vs_true_background
4,trig_L2_calo_eta,True Background,Athena Loose Cut,Athena Loose Cut vs True Background,10384,4309,0.010992,0.001552,1,1,notebooks,vs_true_background


## 5. Statistical Equivalence Evaluation

Evaluates whether compared feature distributions are statistically equivalent or functionally consistent according to each distance metric:

- **JS Divergence (`JS_Equivalent`)**: Returns `True` if symmetric bounded divergence $D_{\text{JS}} \le 2\sigma_{\text{JS}}$ or $D_{\text{JS}} < 0.01$.

In [45]:
# Evaluate statistical equivalence columns across all metrics
df_metrics = DistanceMetricsETL.evaluate_distribution_equivalence(df)

# Inspect sample evaluated records
cols_to_show = [
    "Feature","Reference", "Strategy", "Comparison",
    "JS Divergence", "JS_Equivalent", "JS Uncertainty"
]
df_metrics = df_metrics[cols_to_show]
display((df_metrics[(df_metrics.Feature == "Eratio") &
    (df_metrics.Strategy.str.contains("Model V"))]).sort_values(by="Comparison").drop_duplicates(subset='Comparison'),)

,Feature,Reference,Strategy,Comparison,JS Divergence,JS_Equivalent,JS Uncertainty
69,Eratio,True Background,Model V9 (100 Rings) [T(mu)],Model V9 (100 Rings) [T(mu)] vs True Background,0.217338,False,0.009435
141,Eratio,True Signal,Model V9 (100 Rings) [T(mu)],Model V9 (100 Rings) [T(mu)] vs True Signal,0.019508,False,0.002407
33,Eratio,True Background,Model V9 (100 Rings),Model V9 (100 Rings) vs True Background,0.223816,False,0.010282
105,Eratio,True Signal,Model V9 (100 Rings),Model V9 (100 Rings) vs True Signal,0.019469,False,0.002953
71,Eratio,True Background,Model V9 (50 Rings) [T(mu)],Model V9 (50 Rings) [T(mu)] vs True Background,0.196813,False,0.008957
143,Eratio,True Signal,Model V9 (50 Rings) [T(mu)],Model V9 (50 Rings) [T(mu)] vs True Signal,0.019305,False,0.002834
35,Eratio,True Background,Model V9 (50 Rings),Model V9 (50 Rings) vs True Background,0.207126,False,0.009078
107,Eratio,True Signal,Model V9 (50 Rings),Model V9 (50 Rings) vs True Signal,0.020365,False,0.002672


## 6. Export Distance Metrics to LaTeX Table with Uncertainties

Formats feature comparison distance metrics and their bootstrap uncertainties into LaTeX `$val \pm unc$` representation, retaining `Feature` and `Comparison` while omitting the boolean equivalence flags.

In [48]:
# Format distance metrics and uncertainties into LaTeX table
# Keeps Feature and Comparison, drops equivalence columns, and formats '$val \pm unc$'

formatted_latex_df, latex_code = DistanceMetricsETL.format_latex_table((df_metrics
    [
        (df_metrics.Feature == "Eratio") &
        (df_metrics.Strategy.str.contains("Model V"))
    ]
    ).sort_values(by="Reference").drop_duplicates(subset='Comparison'), precision=4)

# Display formatted table preview
display(formatted_latex_df.head(15))

# Print raw LaTeX code snippet
print("% --- LaTeX Table Snippet ---")
print(latex_code)

,Feature,Comparison,JS Divergence
33,Eratio,Model V9 (100 Rings) vs True Background,$0.2238 \pm 0.0103$
35,Eratio,Model V9 (50 Rings) vs True Background,$0.2071 \pm 0.0091$
69,Eratio,Model V9 (100 Rings) [T(mu)] vs True Background,$0.2173 \pm 0.0094$
71,Eratio,Model V9 (50 Rings) [T(mu)] vs True Background,$0.1968 \pm 0.0090$
105,Eratio,Model V9 (100 Rings) vs True Signal,$0.0195 \pm 0.0030$
107,Eratio,Model V9 (50 Rings) vs True Signal,$0.0204 \pm 0.0027$
141,Eratio,Model V9 (100 Rings) [T(mu)] vs True Signal,$0.0195 \pm 0.0024$
143,Eratio,Model V9 (50 Rings) [T(mu)] vs True Signal,$0.0193 \pm 0.0028$


% --- LaTeX Table Snippet ---
\begin{tabular}{l l c c c c}
\toprule
Feature & Comparison & JS Divergence \\
\midrule
Eratio & Model V9 (100 Rings) vs True Background & $0.2238 \pm 0.0103$ \\
Eratio & Model V9 (50 Rings) vs True Background & $0.2071 \pm 0.0091$ \\
Eratio & Model V9 (100 Rings) [T(mu)] vs True Background & $0.2173 \pm 0.0094$ \\
Eratio & Model V9 (50 Rings) [T(mu)] vs True Background & $0.1968 \pm 0.0090$ \\
Eratio & Model V9 (100 Rings) vs True Signal & $0.0195 \pm 0.0030$ \\
Eratio & Model V9 (50 Rings) vs True Signal & $0.0204 \pm 0.0027$ \\
Eratio & Model V9 (100 Rings) [T(mu)] vs True Signal & $0.0195 \pm 0.0024$ \\
Eratio & Model V9 (50 Rings) [T(mu)] vs True Signal & $0.0193 \pm 0.0028$ \\
\bottomrule
\end{tabular}



# Analysis  strategy metrics

In [27]:
FILE_PATTERN: str = "*_strategy_metrics.csv"

print(f"Configured {len(VALIDATION_FOLDERS)} folders to scan:")
for folder in VALIDATION_FOLDERS:
    print(f" - {folder}")

Configured 1 folders to scan:
 - /home/pmourafr/egamma_master/notebooks/Tables


In [28]:
etl = DistanceMetricsETL(folder_paths=VALIDATION_FOLDERS, file_pattern=FILE_PATTERN)
print(f"Discovered {len(etl.discovered_files)} distance metric CSV files.")

df = etl.extract_and_consolidate()
print(f"Consolidated DataFrame shape: {df.shape}")
df['Strategy'] = df['Strategy'].fillna(df['Model'])
df.drop_duplicates(subset=['Strategy'], inplace=True)
df.drop(columns=["Model"], inplace=True )
df.sort_values(by="Strategy", inplace=True)
df.reset_index(drop=True, inplace=True)

metric_specs = [
    ("PD (Signal Eff)", "PD Uncertainty"),
    ("PF (Fake Rate)", "PF Uncertainty"),
    ("SP Index", "SP Index Uncertainty"),
]

formatted, latex_str = etl.format_table_with_uncertainties(
            df=df,
            pairs=metric_specs,
            id_cols=["Strategy"],
            column_format="l l c c c c",
        )
display(formatted)
print(latex_str)

Discovered 5 distance metric CSV files.
Consolidated DataFrame shape: (10, 14)


,Strategy,PD (Signal Eff),PF (Fake Rate),SP Index
0,Athena Loose Cut,$0.7777 \pm 0.0066$,$0.4150 \pm 0.0048$,$0.6779 \pm 0.0040$
1,Model V9 (100 Rings),$0.8130 \pm 0.0062$,$0.1393 \pm 0.0034$,$0.8367 \pm 0.0035$
2,Model V9 (100 Rings) (Hard Cut),$0.8130 \pm 0.0062$,$0.1393 \pm 0.0034$,$0.8367 \pm 0.0035$
3,Model V9 (100 Rings) [Pileup T(mu)],$0.8150 \pm 0.0061$,$0.1431 \pm 0.0034$,$0.8358 \pm 0.0035$
4,Model V9 (100 Rings) [T(mu)],$0.8150 \pm 0.0061$,$0.1431 \pm 0.0034$,$0.8358 \pm 0.0035$
5,Model V9 (50 Rings),$0.8130 \pm 0.0062$,$0.1458 \pm 0.0035$,$0.8335 \pm 0.0036$
6,Model V9 (50 Rings) [T(mu)],$0.8185 \pm 0.0061$,$0.1526 \pm 0.0035$,$0.8329 \pm 0.0035$


\begin{tabular}{l l c c c c}
\toprule
Strategy & PD (Signal Eff) & PF (Fake Rate) & SP Index \\
\midrule
Athena Loose Cut & $0.7777 \pm 0.0066$ & $0.4150 \pm 0.0048$ & $0.6779 \pm 0.0040$ \\
Model V9 (100 Rings) & $0.8130 \pm 0.0062$ & $0.1393 \pm 0.0034$ & $0.8367 \pm 0.0035$ \\
Model V9 (100 Rings) (Hard Cut) & $0.8130 \pm 0.0062$ & $0.1393 \pm 0.0034$ & $0.8367 \pm 0.0035$ \\
Model V9 (100 Rings) [Pileup T(mu)] & $0.8150 \pm 0.0061$ & $0.1431 \pm 0.0034$ & $0.8358 \pm 0.0035$ \\
Model V9 (100 Rings) [T(mu)] & $0.8150 \pm 0.0061$ & $0.1431 \pm 0.0034$ & $0.8358 \pm 0.0035$ \\
Model V9 (50 Rings) & $0.8130 \pm 0.0062$ & $0.1458 \pm 0.0035$ & $0.8335 \pm 0.0036$ \\
Model V9 (50 Rings) [T(mu)] & $0.8185 \pm 0.0061$ & $0.1526 \pm 0.0035$ & $0.8329 \pm 0.0035$ \\
\bottomrule
\end{tabular}



# Analysis _points_efficiency

In [ ]:
FILE_PATTERN: str = "*_points_efficiency.csv"

print(f"Configured {len(VALIDATION_FOLDERS)} folders to scan:")
for folder in VALIDATION_FOLDERS:
    print(f" - {folder}")